# 2. Translation and merge

Takes the long files built by `Compendium_1_Long_Files.ipynb` and produces one
English file per chapter:

1. **Translate** `merged_long_files\<Chapter>_AR.xlsx` into English.
2. **Append** `merged_long_files\<Chapter>_EN.xlsx` if that chapter also had
   English questionnaires - those rows are already English and need no
   translation.
3. **Save** the combined result as `COMPENDIUM-ARAB SOCIETY\<Chapter>_EN.xlsx`.

The calculated indicators are **notebook 3's** job, not this one's.

The combined file is written to a third location and rebuilt from scratch each
run, so `merged_long_files` stays purely what notebook 1 put there and
re-running can never append the same rows twice.

Anything the dictionary has no entry for passes through untranslated and is
listed at the end, ready for `export_untranslated()` -> Claude Code ->
`update_dictionary()`.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration - paths and chapters.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# Every long file lives here - one folder, both languages. The _AR / _EN suffix
# is already in each filename, so a folder per language only meant two places to
# look.
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"

# Questionnaire folders are named <prefix><LANGUAGE>. The suffix IS the language,
# so nothing here has to list them - add a folder and it is picked up.
QUESTIONNAIRE_PREFIX = "datacollector_received_quest_"

# One long file per chapter per language lands here.


# Leave CHAPTERS as None to process every chapter found on disk. Set an explicit
# list to restrict one run, e.g. CHAPTERS = ["Poverty"].
CHAPTERS = None

LANGUAGES = ["AR", "EN"]

# The language the questionnaires mostly arrive in, and the one we translate to.
SOURCE_LANGUAGE = "AR"
TARGET_LANGUAGE = "EN"


def discover_chapters():
    """Chapters that notebook 1 produced a long file for, in either language."""
    names = set()
    for language in LANGUAGES:
        folder = LONG_FILES_PATH
        if not folder.exists():
            continue
        suffix = f"_{language}.xlsx"
        for path in folder.glob(f"*{suffix}"):
            names.add(path.name[: -len(suffix)])
    return sorted(names)


def chapters_to_process():
    """CHAPTERS if it was set, otherwise whatever notebook 1 produced."""
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters with a long file: {found}")
    if not found:
        logger.warning("No long files found - run notebook 1 first.")
    return found


# ---------------------------------------------------------------------------
# Everything the pipeline finds wrong with the SOURCE DATA is collected here,
# from all four notebooks. Each owns a section and rewrites only its own, so the
# file always reflects the latest run of each step whatever order they ran in.
INCONSISTENCY_LOG_PATH = COMPENDIUM_PATH / "pipeline_inconsistencies.txt"


def save_inconsistencies(section, records):
    """Write this notebook's findings into the shared file, replacing its own
    section. `records` is a list of dicts; the keys become the columns."""
    marker = f"### {section} ###"
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")

    body = [marker, f"    last run {stamp}", ""]
    if not records:
        body += ["    Nothing found.", ""]
    else:
        frame = pd.DataFrame(records)
        for kind, group in frame.groupby("kind", sort=False):
            body.append(f"  {kind.upper()}  ({len(group)})")
            for _, row in group.iterrows():
                def show(value):
                    # A record without a year forces that column to float, so
                    # 2010 would otherwise print as "2010.0".
                    if isinstance(value, float) and float(value).is_integer():
                        return str(int(value))
                    return str(value)
                where = " \u00b7 ".join(
                    show(row[f]) for f in ("chapter", "file", "sheet", "country", "year", "sex")
                    if f in row and pd.notna(row[f]) and str(row[f]) != "")
                body.append(f"      {where}" if where else "      -")
                body.append(f"          {row['detail']}")
            body.append("")

    section_text = "\n".join(body)

    header = [
        "PIPELINE INCONSISTENCIES",
        "",
        "Everything the notebooks found wrong with the SOURCE DATA - not with the",
        "code. A figure that disagrees with itself cannot be reconciled downstream;",
        "the country that reported it is the only place it can be corrected.",
        "",
        "Each notebook rewrites its own section on every run.",
        "=" * 78,
        "",
    ]

    # Read what is already there and split it into sections, so this one can
    # replace its own and the file be rebuilt in step order. Appending instead
    # left the sections in whatever order the notebooks last ran, which reads as
    # though steps had been skipped.
    sections = {}
    if INCONSISTENCY_LOG_PATH.exists():
        existing = INCONSISTENCY_LOG_PATH.read_text(encoding="utf-8")
        parts = re.split(r"^### (.+?) ###$", existing, flags=re.M)
        for name, text in zip(parts[1::2], parts[2::2]):
            sections[name] = f"### {name} ###{text.rstrip()}"
    sections[section] = section_text

    body = "\n\n".join(sections[name] for name in sorted(sections))
    INCONSISTENCY_LOG_PATH.write_text(
        "\n".join(header).rstrip("\n") + "\n\n" + body + "\n", encoding="utf-8")
    return INCONSISTENCY_LOG_PATH, len(records)


## Load the translation dictionary

One dictionary, Arabic to English - no reverse is built, because the
questionnaires arrive in Arabic and English is what we translate into.


In [ ]:
"""
CELL: Load translation dict.xlsx - one dictionary, Arabic to English.
"""


def load_dictionary():
    """Reads translation dict.xlsx into:

      DICTIONARY_AR_TO_EN  (column_map, value_map)
          column_map = {Arabic column name: English column name}
          value_map  = {Arabic column name: {Arabic value: English value}}

      ENGLISH_VOCABULARY  (column_names, values_by_column)
          the English column names and values the file knows about.

    Only the Arabic-to-English direction is built, because the questionnaires
    arrive in Arabic and English is what we translate into.

    ENGLISH_VOCABULARY is not a reverse dictionary - it maps nothing. It is just
    the list of correct English spellings, so an English questionnaire's
    misspelled labels can be fuzzy-matched against the right words too.
    """
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            arabic: english
            for arabic, english in zip(rows["val_ar"], rows["val_en"])
            if pd.notna(arabic)
        }

    english_columns = {}
    english_values = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_columns[english_column] = english_column
        english_values[english_column] = {
            str(v): str(v) for v in rows["val_en"].dropna().unique()
        }

    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return (column_map, value_map), (english_columns, english_values), chapter_to_arabic


DICTIONARY_AR_TO_EN, ENGLISH_VOCABULARY, CHAPTER_TO_ARABIC = load_dictionary()


def vocabulary(language):
    """The known column names and values for a language, as
    (column_names, values_by_column) - what misspellings are matched against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else ENGLISH_VOCABULARY


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names, Arabic -> English")


## `translate()`

Swaps every Arabic value for its English equivalent, then renames the column.
Anything the dictionary has no entry for is left exactly as it is.


In [ ]:
"""
CELL: translate() - Arabic to English, using the one dictionary.
"""


def translate(table):
    """Swaps every Arabic value for its English equivalent, then renames the
    column to its English name.

    Values are replaced first and the column renamed second, because the value
    lookup is keyed by the column's ORIGINAL Arabic name - renaming first would
    lose it. Anything the dictionary has no entry for is left exactly as it is,
    and is picked up afterwards by find_untranslated().
    """
    column_map, value_map = DICTIONARY_AR_TO_EN

    table = table.copy()
    for column in list(table.columns):
        if column in value_map:
            table[column] = table[column].replace(value_map[column])
        if column in column_map:
            table = table.rename(columns={column: column_map[column]})
    return table


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it."""
    return any("\u0600" <= character <= "\u06ff" for character in str(text))


## Finding and filling the dictionary's gaps

### Letting Claude Code close the gaps for you

You do not have to shuttle the file back and forth. Ask once:

> **"run the pipeline and fill any dictionary gaps"**

Claude Code runs the notebook, translates whatever the dictionary did not know,
writes it back into `translation dict.xlsx` (taking a backup first), re-runs to
confirm the gap list is empty, and reports what it added. The protocol is
recorded in `CLAUDE.md`, so it does not need explaining again each session.

Doing it by hand is still the same three calls: `export_untranslated(REPORTS)`
-> fill in the blank column -> `update_dictionary(filled)`.


In [ ]:
"""
CELL: find_untranslated() - what the dictionary could not translate.
"""


def find_untranslated(arabic_table, translated_table):
    """Values that came out of translate() unchanged and are still Arabic.

    A value with no dictionary entry is copied through untouched, so comparing
    the table before and against after finds them exactly. Values already in
    Latin script are not gaps - plenty of Arabic questionnaires cite their
    source in English ("MICS 2022", a URL) and those are correct as they stand.
    """
    column_map, _ = DICTIONARY_AR_TO_EN
    gaps = []

    for arabic_column, english_column in zip(arabic_table.columns, translated_table.columns):
        before = arabic_table[arabic_column]
        after = translated_table[english_column]
        pairs = pd.DataFrame({"val_ar": before, "val_en": after}).dropna()

        for (arabic_value, english_value), count in pairs.groupby(["val_ar", "val_en"]).size().items():
            if str(arabic_value).strip() != str(english_value).strip():
                continue                      # translated fine
            if not looks_arabic(arabic_value):
                continue                      # already English
            gaps.append({
                "col_ar": arabic_column, "col_en": english_column,
                "val_ar": arabic_value, "val_en": None, "rows": count,
            })
    return gaps


def export_untranslated(reports, file_name="untranslated_values.xlsx"):
    """Writes every gap found during the run to one Excel file, shaped like
    translation dict.xlsx so a filled-in row can go straight back into it."""
    rows = [gap for report in reports for gap in report["untranslated"]]
    if not rows:
        logger.info("Nothing untranslated - the dictionary covered every value.")
        return pd.DataFrame()

    gaps = (pd.DataFrame(rows)
            .drop_duplicates(subset=["col_ar", "val_ar"])
            .sort_values(["col_en", "val_ar"])
            .reset_index(drop=True))
    path = COMPENDIUM_PATH / file_name
    gaps.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"Saved {path.name}: {len(gaps):,} value(s) with no dictionary entry. "
                f"Fill in val_en, then call update_dictionary().")
    return gaps


def update_dictionary(filled, backup=True):
    """Appends reviewed translations to translation dict.xlsx.

    `filled` is the exported table with val_en filled in (a DataFrame, or a path
    to the saved file). Rows missing either side are skipped, and an Arabic
    value the dictionary already has is left alone - so running this twice
    changes nothing the second time. A timestamped backup is written first,
    because this edits the project's source of truth.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    new_rows = filled[needed].dropna()
    new_rows = new_rows[(new_rows["val_ar"].astype(str).str.strip() != "")
                        & (new_rows["val_en"].astype(str).str.strip() != "")]
    if new_rows.empty:
        logger.warning("No completed rows to add - is val_en filled in?")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already_there = set(zip(dictionary["col_ar"], dictionary["val_ar"]))
    to_add = new_rows[~new_rows.apply(
        lambda r: (r["col_ar"], r["val_ar"]) in already_there, axis=1)]
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx")
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    # Mark what the pipeline added, so a row typed by hand and a row translated
    # by Claude Code can be told apart later. Originals keep a blank status.
    to_add = to_add.reindex(columns=dictionary.columns)
    if "status" in to_add.columns:
        to_add["status"] = "updated"

    updated = pd.concat([dictionary, to_add], ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    logger.info(f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
                f"({len(dictionary):,} -> {len(updated):,}). Re-run to use them.")
    return updated


def calculated_labels():
    """Every label this notebook INVENTS, as (English column, English value).

    These come from the calculations, not from any questionnaire, so the
    dictionary can only ever learn them from here. Derived from the same
    constants the calculations use, so adding a calculation cannot leave this
    list behind.
    """
    labels = [("Indicator", SEX_RATIO_TITLE), ("Indicator", AGE_SHARE_TITLE)]
    labels += [("Age Group", group) for group in AGE_GROUPS]
    return labels


def check_calculated_labels():
    """Which invented labels the dictionary does not know yet.

    Returned in the gap shape used elsewhere, except that here it is the ARABIC
    side that is blank: the English is what this notebook chose, and the Arabic
    is what has to be supplied before notebook 3 can render these rows.
    """
    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    known = set(zip(dictionary["col_en"].astype(str).str.strip(),
                    dictionary["val_en"].astype(str).str.strip()))

    column_map, _ = DICTIONARY_AR_TO_EN
    english_to_arabic_column = {en: ar for ar, en in column_map.items()}

    missing = []
    for english_column, label in calculated_labels():
        if (english_column, str(label).strip()) in known:
            continue
        missing.append({
            "col_ar": english_to_arabic_column.get(english_column, english_column),
            "col_en": english_column,
            "val_ar": None,          # <- to be filled in
            "val_en": label,
            "rows": None,
        })
    return pd.DataFrame(missing)


def export_calculated_labels(file_name="new_labels_to_translate.xlsx"):
    """Writes the invented labels the dictionary does not know to their own
    file, kept separate from the questionnaire gaps because the blank column is
    the other one."""
    missing = check_calculated_labels()
    if missing.empty:
        logger.info("The dictionary knows every label the calculations invent.")
        return missing
    path = COMPENDIUM_PATH / file_name
    missing.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"Saved {path.name}: {len(missing)} invented label(s) with no Arabic. "
                f"Fill in val_ar, then call update_dictionary().")
    return missing


## `build_chapter()`

Translate the Arabic long file, append the native English one if it exists, and
save the combined result. Every step is logged, so the run output shows
exactly which chapters had an English file appended.


In [ ]:
"""
CELL: build_chapter() - translate, append the native English file, save.
"""


def build_chapter(chapter):
    """Produces COMPENDIUM-ARAB SOCIETY/<chapter>_EN.xlsx and reports what went
    into it.

    Returns a dict describing the run, so the cell below can print one table
    showing which chapters had an English long file appended and which did not.
    """
    arabic_path = LONG_FILES_PATH / f"{chapter}_AR.xlsx"
    english_path = LONG_FILES_PATH / f"{chapter}_EN_questionnaires.xlsx"
    output_path = LONG_FILES_PATH / f"{chapter}_EN.xlsx"

    report = {"chapter": chapter, "translated_rows": 0, "appended_rows": 0,
              "total_rows": 0, "appended_from": None, "untranslated": []}

    parts = []

    if arabic_path.exists():
        arabic_table = pd.read_excel(arabic_path, engine="openpyxl")
        translated = translate(arabic_table)
        parts.append(translated)
        report["translated_rows"] = len(translated)
        logger.info(f"  {chapter}: translated {len(translated):,} row(s) from "
                    f"merged_long_files\\{arabic_path.name}")
        report["untranslated"] = find_untranslated(arabic_table, translated)
    else:
        logger.info(f"  {chapter}: no Arabic long file")

    if english_path.exists():
        english_table = pd.read_excel(english_path, engine="openpyxl")
        parts.append(english_table)
        report["appended_rows"] = len(english_table)
        report["appended_from"] = f"merged_long_files\\{english_path.name}"
        logger.info(f"  {chapter}: APPENDED {len(english_table):,} row(s) from "
                    f"{report['appended_from']} (already English, not translated)")
    else:
        logger.info(f"  {chapter}: no English long file to append")

    if not parts:
        logger.warning(f"  {chapter}: nothing to build - run notebook 1 first")
        return report

    # Concatenate, not merge side-by-side: both parts are the same long shape,
    # so this stacks the English-sourced rows under the translated ones.
    combined = pd.concat(parts, ignore_index=True)
    combined.to_excel(output_path, index=False, engine="openpyxl")
    report["total_rows"] = len(combined)
    logger.info(f"  {chapter}: saved {output_path.name} ({len(combined):,} rows)")
    return report


## Run - translate, append, save


In [ ]:
"""
CELL: Main run - build every chapter's combined English file.
"""
print(f"Translating {SOURCE_LANGUAGE} -> {TARGET_LANGUAGE}")
print(f"  reading   {LONG_FILES_PATH}")
print(f"  appending <Chapter>_EN_questionnaires.xlsx, where one exists")
print(f"  writing   {LONG_FILES_PATH}\\<Chapter>_EN.xlsx\n")

REPORTS = []
chapters = chapters_to_process()
total_chapters = len(chapters)
for i, chapter in enumerate(chapters, start=1):
    bar = "#" * i + "-" * (total_chapters - i)
    print(f"[{bar}] chapter {i}/{total_chapters}: {chapter}")
    REPORTS.append(build_chapter(chapter))

print("\n" + "=" * 78)
print("WHAT WENT INTO EACH FILE")
print("=" * 78)
print(f"{'Chapter':<12}{'translated AR':>15}{'appended EN':>14}{'total':>12}   appended from")
for r in REPORTS:
    appended = r["appended_from"] or "-  (no English long file)"
    print(f"{r['chapter']:<12}{r['translated_rows']:>15,}{r['appended_rows']:>14,}"
          f"{r['total_rows']:>12,}   {appended}")

with_english = [r["chapter"] for r in REPORTS if r["appended_rows"]]
print(f"\nChapters that had an English long file appended: "
      f"{with_english if with_english else 'none'}")

gap_count = len({(g["col_ar"], g["val_ar"]) for r in REPORTS for g in r["untranslated"]})
print(f"Distinct values with no dictionary entry: {gap_count}")
if gap_count:
    print("Run export_untranslated(REPORTS), ask Claude Code to fill in val_en,")
    print("then update_dictionary() - and run this cell again.")

# ------------------------------------------------------- the shared log
records = []
seen = set()
for report in REPORTS:
    for gap in report["untranslated"]:
        key = (gap["col_ar"], gap["val_ar"])
        if key in seen:
            continue
        seen.add(key)
        records.append({
            "kind": "no English translation", "chapter": report["chapter"],
            "detail": f"{gap['col_en']}: {gap['val_ar']!r} appears in "
                      f"{gap['rows']} row(s) and passes through untranslated",
        })

written, count = save_inconsistencies("2. TRANSLATION", records)
print(f"{count} inconsistency(ies) recorded in {written.name}")


## Gaps

`export_untranslated(REPORTS)` writes every value the dictionary could not
translate to `untranslated_values.xlsx`, with `val_en` blank. Ask Claude Code to
fill it in, then `update_dictionary()` writes the rows back into
`translation dict.xlsx` and the next run translates them.


In [ ]:
"""
CELL: Write the gap file for the run above.
"""
UNTRANSLATED = export_untranslated(REPORTS)
UNTRANSLATED.head(20)
